<a href="https://colab.research.google.com/github/Abem-S/ai_future/blob/main/edge_ai_prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import numpy as np
import time
import os

In [6]:
# --- Configuration for Simulation ---
IMG_SIZE = 128
NUM_CLASSES = 2 # Recyclable, Non-Recyclable
BATCH_SIZE = 32
EPOCHS = 3
TRAIN_SAMPLES = 1000
TEST_SAMPLES = 200
MODEL_FILENAME = 'recycling_classifier.h5'
TFLITE_FILENAME = 'recycling_classifier_quantized.tflite'

print(f"Starting Edge AI Prototype Simulation...")

Starting Edge AI Prototype Simulation...


In [7]:
# Step 1 & 2: Data Simulation and Setup
def generate_synthetic_data(num_samples):
    """Generates random image data and corresponding labels."""
    print(f"Generating {num_samples} synthetic samples...")
    # Simulate image data (normalized to [0, 1])
    images = np.random.rand(num_samples, IMG_SIZE, IMG_SIZE, 3).astype(np.float32)
    # Simulate one-hot encoded labels (e.g., [1, 0] or [0, 1])
    labels = np.random.randint(0, NUM_CLASSES, size=(num_samples,))
    labels = keras.utils.to_categorical(labels, num_classes=NUM_CLASSES)
    return images, labels

x_train, y_train = generate_synthetic_data(TRAIN_SAMPLES)
x_test, y_test = generate_synthetic_data(TEST_SAMPLES)

print(f"Training data shape: {x_train.shape}, Test data shape: {x_test.shape}")

Generating 1000 synthetic samples...
Generating 200 synthetic samples...
Training data shape: (1000, 128, 128, 3), Test data shape: (200, 128, 128, 3)


In [8]:
# Step 3: Model Training (Transfer Learning with MobileNetV2)
print("\n--- Step 3: Training Keras Model (Transfer Learning) ---")

# Load MobileNetV2 base model with pre-trained ImageNet weights
# This is the line most likely to hang or fail without proper setup/restart
try:
    base_model = MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False, # Crucial: Don't include the large default classifier head
        weights='imagenet'
    )
    base_model.trainable = False # Freeze the base weights
except Exception as e:
    print(f"ERROR: Failed to load MobileNetV2 weights. Check your TensorFlow installation and restart the runtime.")
    raise e


--- Step 3: Training Keras Model (Transfer Learning) ---


In [9]:
# Add our custom classification layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

In [10]:
# Final Keras Model
model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [11]:
print("Training model...")
# Training on synthetic data
history = model.fit(x_train, y_train,
                    batch_size=BATCH_SIZE,
                    epochs=EPOCHS,
                    validation_data=(x_test, y_test),
                    verbose=0)

model.save(MODEL_FILENAME)
print(f"Keras Model saved to: {MODEL_FILENAME}")

Training model...


Keras Model saved to: recycling_classifier.h5


In [12]:
# Step 4: Initial Evaluation
print("\n--- Step 4: Initial Keras Model Evaluation ---")
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Keras Baseline Accuracy: {accuracy:.4f}")
print(f"Keras Model Size: {os.path.getsize(MODEL_FILENAME) / (1024*1024):.2f} MB")


--- Step 4: Initial Keras Model Evaluation ---
Keras Baseline Accuracy: 0.5350
Keras Model Size: 11.01 MB


In [13]:
# Step 5: Model Conversion to TFLite (Quantization)
print("\n--- Step 5: TFLite Conversion and Quantization ---")

# Quantization requires a representative dataset for calibration
def representative_data_gen():
    for input_value in x_train[:100]:
        yield [np.expand_dims(input_value, axis=0)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT] # Enable default quantization
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_model = converter.convert()

with open(TFLITE_FILENAME, 'wb') as f:
    f.write(tflite_model)

tflite_size_mb = len(tflite_model) / (1024 * 1024)
print(f"TFLite Model saved to: {TFLITE_FILENAME}")
print(f"TFLite Quantized Model Size: {tflite_size_mb:.2f} MB (Significant size reduction achieved)")


--- Step 5: TFLite Conversion and Quantization ---
Saved artifact at '/tmp/tmpbjldn1ub'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='keras_tensor_628')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  132328709665616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132328709667536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132328709667728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132328709668304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132328709666960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132328709668496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132328709667344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132328709667152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132328709668112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132328709666000: TensorSpec(

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


TFLite Model saved to: recycling_classifier_quantized.tflite
TFLite Quantized Model Size: 2.74 MB (Significant size reduction achieved)


In [14]:
# Step 6: TFLite Inference Test and Latency Check
print("\n--- Step 6: TFLite Inference and Latency Check ---")

interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

# Prepare a single test image (must be converted to the model's required uint8 input type)
sample_image = x_test[0:1] # Get first image
# Scale the input data to [0, 255] and convert to uint8 for quantized model
input_data = (sample_image * 255).astype(np.uint8)



--- Step 6: TFLite Inference and Latency Check ---


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [15]:
# Run Inference (Latency Simulation)
start_time = time.time()
interpreter.set_tensor(input_details['index'], input_data)
interpreter.invoke()
output_data = interpreter.get_tensor(output_details['index'])
end_time = time.time()

# Calculate latency
latency_ms = (end_time - start_time) * 1000

print(f"Inference Latency for 1 image: {latency_ms:.3f} ms")
print(f"TFLite Prediction Output (scores): {output_data[0]}")
print("Prototype successfully executed. The TFLite model is ready for Edge deployment.")

Inference Latency for 1 image: 12.048 ms
TFLite Prediction Output (scores): [ 52 204]
Prototype successfully executed. The TFLite model is ready for Edge deployment.
